# ManoBuddhi F4: real models, GPU telemetry, GSM8K baselines

Runs the fast model (Qwen2.5-1.5B-Instruct) and the slow model (Qwen2.5-7B-Instruct) on Kaggle's 2x T4 in float16, measures every call (latency, time to first token, GPU-seconds, energy), trains the hidden-state probe, and computes the three week-3 baselines: **always-slow**, **log-prob confidence cascade**, **probe-only**.

**Keep this notebook, its outputs and any dataset made from them PRIVATE** until the Indian provisional is filed.

**Before running**
1. Settings (right panel): Accelerator **GPU T4 x2**, Internet **On** (needs phone verification).
2. Add-ons → Secrets → add `GITHUB_TOKEN` (fine-grained, repo `manobuddhi` only, Contents read-only) and tick it for this notebook.
3. Pick `PROFILE` below: run `smoke` first (about 20 min, interactive), then `checkpoint` with **Save Version → Save & Run All (Commit)** so it runs in the background (about 6–8 h).

**Resume after the 12-hour limit:** open the finished version's Output → *New Dataset* (private), add that dataset to this notebook, set `RESUME_FROM` to its folder name and run again. Collection skips items already done.

**Afterwards:** download `f4_<RUN>.zip` from Output, unzip it into the project's `results\` folder on your PC, and run `python scripts\import_results.py results\<RUN>`.

In [ ]:
# ---- settings -------------------------------------------------------------
RUN = "r1"                 # results/<RUN>; use a new name for different models
PROFILE = "smoke"          # smoke | checkpoint | full
RESUME_FROM = None         # e.g. "manobuddhi-f4-r1-output" (a private dataset made from an earlier version)

MODE = "small_large"       # Config B. "single_model_nothink_think" = Config A (one Qwen3 model)
FAST_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
SLOW_MODEL = "Qwen/Qwen2.5-7B-Instruct"   # Config A: e.g. "Qwen/Qwen3-4B"
SLOW_QUANT = "none"        # none = float16 split over both GPUs; "4bit" = GPU 1 only (needs bitsandbytes)
SLOW_MAX_MEMORY = {0: "4GiB", 1: "12GiB"}
RUN_TRAPS = True           # fast model on Hagendorff trap items: probe vs log-prob AUROC (O1)

PROFILES = {               # items per split
    "smoke":      dict(probe_train=64,   calib=20,  test=20,   traps=20),
    "checkpoint": dict(probe_train=2000, calib=300, test=500,  traps=240),
    "full":       dict(probe_train=4000, calib=500, test=1319, traps=240),
}
N = PROFILES[PROFILE]
if PROFILE == "smoke":
    RUN = RUN + "-smoke"
print(RUN, N)

In [ ]:
!nvidia-smi --query-gpu=index,name,memory.total,driver_version --format=csv

In [ ]:
# ---- clone the private repo with the token from Kaggle Secrets (never printed) ----
import os, subprocess
from kaggle_secrets import UserSecretsClient

REPO = "/tmp/manobuddhi"   # code stays out of the saved Output; only results are saved
URL = "github.com/AniveshGuptaOfficial/manobuddhi.git"
if not os.path.exists(REPO):
    tok = UserSecretsClient().get_secret("GITHUB_TOKEN")
    p = subprocess.run(["git", "clone", "--depth", "1", f"https://x-access-token:{tok}@{URL}", REPO],
                       capture_output=True, text=True)
    del tok
    if p.returncode != 0:
        raise RuntimeError("git clone failed: check the GITHUB_TOKEN secret is attached and has "
                           "Contents read access to manobuddhi. " + p.stderr.split("@")[-1][-300:])
    subprocess.run(["git", "-C", REPO, "remote", "set-url", "origin", "https://" + URL], check=True)
print(subprocess.run(["git", "-C", REPO, "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

In [ ]:
!pip install -q -U "transformers>=4.51,<5" "accelerate>=0.30" nvidia-ml-py scikit-learn
if SLOW_QUANT != "none":
    !pip install -q -U bitsandbytes

In [ ]:
import shutil, sys
from pathlib import Path

sys.path.insert(0, REPO)
os.chdir(REPO)
OUT = Path("/kaggle/working/results")
run_path = OUT / RUN
if RESUME_FROM:
    src = Path("/kaggle/input") / RESUME_FROM / "results" / RUN
    if src.exists() and not run_path.exists():
        shutil.copytree(src, run_path)
        print("resumed from", src)

from engine.results import RunDir
run = RunDir(run_path)
print(run.path, run.manifest().get("collections", {}))

In [ ]:
# ---- load models (about 3-5 min the first time: downloads ~18 GB) ----
from engine.collect import collect, load_pair
from engine import pipeline

if "pair" not in globals():   # re-running this cell must not load a second copy (GPU out of memory)
    pair = load_pair(MODE, fast_model=FAST_MODEL, slow_model=SLOW_MODEL, slow_quant=SLOW_QUANT,
                     slow_max_memory=SLOW_MAX_MEMORY)
import json; print(json.dumps(pair.describe(), indent=1)[:1500])
!nvidia-smi --query-gpu=index,memory.used,memory.total --format=csv

In [ ]:
# ---- sanity check: 3 items, print what was measured ----
chk = RunDir(OUT / (RUN + "-check"))
collect(chk, pair, "gsm8k", "test", n=3)
for r in chk.load_records("gsm8k", "test"):
    f, s = r["fast"], r["slow"]
    print(r["id"], "gold", r["item"]["gold"], "| fast", r["fast_correct"], "slow", r["slow_correct"])
    for name, g in (("fast", f), ("slow", s)):
        print(f"  {name}: {g['completion_tokens']} tok, TTFT {g['ttft_ms']:.0f} ms, {g['latency_ms']:.0f} ms, "
              f"{g['gpu_seconds']:.2f} GPU-s, {g['energy_j']} J, util {g['gpu_util_pct']}, conf {g['conf_seq']}")
    print("  fast tail:", f["text"][-120:].replace("\n", " "))
    print("  probe pass:", {k: r["probe"][k] for k in ("latency_ms", "gpu_seconds")})

### 1. Probe training data: fast model only, batched (timing not reported)

In [ ]:
collect(run, pair, "gsm8k", "probe_train", n=N["probe_train"], paths=("fast",), batch_size=16)

### 2. Calibration set (thresholds are chosen here only) and 3. test set: fast + slow, batch 1, timed

In [ ]:
collect(run, pair, "gsm8k", "calib", n=N["calib"])

In [ ]:
collect(run, pair, "gsm8k", "test", n=N["test"])

### 4. Trap items (optional): does the probe catch confident errors that log-prob misses?

In [ ]:
if RUN_TRAPS:
    collect(run, pair, "traps_hagendorff", "test", n=N["traps"], paths=("fast",))

### 5. Probe, scores, baselines (CPU)

In [ ]:
pipeline.train(run)
pipeline.score(run)
res = pipeline.baselines(run)

In [ ]:
# ---- package results for download (Output tab) ----
archive = shutil.make_archive(f"/kaggle/working/f4_{RUN}", "zip", OUT, RUN)
print(archive, round(os.path.getsize(archive) / 2**20, 1), "MB")